In [1]:
from dotenv import load_dotenv
import os
import json
import anthropic
from datetime import datetime
from pathlib import Path
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal
from langchain_anthropic import ChatAnthropic
from langchain_classic.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain_classic import LLMChain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = "claude-sonnet-4-6"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
llm = ChatAnthropic(model=MODEL, anthropic_api_key=ANTHROPIC_API_KEY, max_tokens=250)

Path("output").mkdir(exist_ok=True)

In [11]:
from typing import Literal
from pydantic import BaseModel, field_validator

In [12]:
class IntakeRecord(BaseModel):
    query_type: Literal[
        "sar_str_filing",
        "kyc_exception",
        "transaction_dispute",
        "sanctions_match",
        "regulatory_examination",
        "fraud_investigation",
        "general_enquiry"
    ]

    regulation_ref: Literal[
        "FATF_R20",
        "BSA_AML",
        "FCA_SYSC",
        "RBI_PMLA",
        "MAS_AML",
        "PCI_DSS",
        "OFAC_SDN",
        "other"
    ]

    jurisdiction: Literal[
        "india",
        "uk",
        "singapore",
        "usa",
        "eu",
        "multi_jurisdictional",
        "general"
    ]

    urgency: Literal[
        "routine",
        "standard",
        "urgent",
        "critical"
    ]

    submitting_team: str

    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")

        return v.strip().title()

    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person_or_identifier(cls, v):
        # Reject likely individual names
        invalid_values = [
            "john",
            "rajesh",
            "suresh",
            "ravi",
            "smith"
        ]

        if v.lower() in invalid_values:
            raise ValueError(
                "submitting_team must contain a team or business unit, not a person's name."
            )
        # Reject identifiers that contains numbers instead of names
        if any(char.isdigit() for char in v):
            raise ValueError(
                "submitting_team must not contain account numbers, case references, "
                "or other numeric identifiers."
            )

        return v

In [13]:
INTAKE_TOOL = {
    "name": "extract_intake_fields",
    "description": (
        "Extract structured AML and compliance intake fields from a "
        "financial services compliance query."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query_type": {
                "type": "string",
                "enum": [
                    "sar_str_filing",
                    "kyc_exception",
                    "transaction_dispute",
                    "sanctions_match",
                    "regulatory_examination",
                    "fraud_investigation",
                    "general_enquiry"
                ]
            },
            "regulation_ref": {
                "type": "string",
                "enum": [
                    "FATF_R20",
                    "BSA_AML",
                    "FCA_SYSC",
                    "RBI_PMLA",
                    "MAS_AML",
                    "PCI_DSS",
                    "OFAC_SDN",
                    "other"
                ]
            },
            "jurisdiction": {
                "type": "string",
                "enum": [
                    "india",
                    "uk",
                    "singapore",
                    "usa",
                    "eu",
                    "multi_jurisdictional",
                    "general"
                ]
            },
            "urgency": {
                "type": "string",
                "enum": [
                    "routine",
                    "standard",
                    "urgent",
                    "critical"
                ]
            },
            "submitting_team": {
                "type": "string"
            }
        },
        "required": [
            "query_type",
            "regulation_ref",
            "jurisdiction",
            "urgency",
            "submitting_team"
        ]
    }
}

In [14]:
EXTRACTION_SYSTEM = """
You are SmartIntake, a financial crime compliance intake specialist.

Use the extract_intake_fields tool to extract structured AML and compliance
information from the user's query.

RULES:

- Think step by step about the regulatory jurisdiction and compliance context
  before selecting enum values.
- Never infer urgency from words such as "urgent", "ASAP", or "please".
- Determine urgency only from filing deadlines, response deadlines, or
  regulatory obligations.
- If deadline information is unclear, leave urgency missing and ask.
- submitting_team must be a team or business unit, never an individual's name.
- Do not extract customer names, customer identifiers, account numbers,
  transaction references, or investigation IDs.
- If a person's name is provided instead of a team, leave submitting_team missing.
- Use "other" when regulation_ref cannot be confidently identified.
- Never invent a regulation_ref.
- Never ask questions that could constitute tipping-off a customer about an
  ongoing investigation.
- If the request is not related to AML, compliance, fraud, sanctions,
  KYC, regulatory examinations, or financial crime operations, do NOT call
  the tool.

Instead respond:

"I handle AML and compliance intake only. Please describe your compliance,
fraud, sanctions, KYC, or regulatory query."

URGENCY GUIDANCE:

- routine:
  No stated deadline and low operational impact.

- standard:
  Normal business processing required.

- urgent:
  Regulatory response or investigation requires action within days.

- critical:
  Legally binding filing or response deadline within 24 hours, or a
  severe compliance risk requiring immediate action.

EXAMPLE 1:

User:
"Transaction monitoring flagged a structuring pattern on a business account.
We need to file a SAR with FinCEN. The 30-day filing window closes tomorrow."

→ call extract_intake_fields with:

query_type="sar_str_filing"
regulation_ref="BSA_AML"
jurisdiction="usa"
urgency="critical"
submitting_team=[ask]

EXAMPLE 2:

User:
"KYC Operations team. We have a PEP match on a new UK customer onboarding.
Enhanced due diligence is required before activation."

→ call extract_intake_fields with:

query_type="kyc_exception"
regulation_ref="FCA_SYSC"
jurisdiction="uk"
urgency="urgent"
submitting_team="KYC Operations"

EXAMPLE 3:

User:
"Hi, we have a new OFAC sanctions hit on a customer in our Singapore business."

→ call extract_intake_fields with:

query_type="sanctions_match"
regulation_ref="OFAC_SDN"
jurisdiction="singapore"
urgency=[ask]
submitting_team=[ask]

EXAMPLE 4:

User:
"My name is Ravi and we need to report suspicious activity."

→ call extract_intake_fields with:

query_type="sar_str_filing"
regulation_ref=[determine]
jurisdiction=[determine]
urgency=[ask]
submitting_team=[ask]

Do not use Ravi as submitting_team because it is an individual's name.
"""
CONFIRMATION_SYSTEM = """
You are SmartIntake.

Generate a formal compliance acknowledgement.

Requirements:
- Reference the query type and regulation formally.
- State the jurisdiction.
- State the urgency tier.
- Name the submitting team.
- Keep the response professional and audit-friendly.
- End with:

'Please confirm or correct before this record is saved.'
"""
FOLLOWUP_SYSTEM = """
You are SmartIntake.

The user has already provided part of the required compliance intake
information.

Ask only for the missing fields listed.

Rules:

- Ask for only one missing field at a time.
- Do not ask for fields that have already been provided.
- Use conversation history to avoid repeating questions.
- Never ask for customer names, account numbers, transaction IDs,
  case references, or investigation identifiers.
- If urgency is missing, ask for the filing or response deadline,
  not the user's opinion of urgency.
- Be concise and professional.
"""

In [24]:
def extract_with_tool(conversation_history: list) -> dict | None:
    """Call Claude with the AML compliance tool schema."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system=EXTRACTION_SYSTEM,
        tools=[INTAKE_TOOL],
        tool_choice={"type": "auto"},
        messages=conversation_history
    )

    if response.stop_reason == "tool_use":
        for block in response.content:
            if block.type == "tool_use":
                return block.input

    return None


def generate_confirmation(record: IntakeRecord) -> str:
    """Generate a compliance acknowledgement."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        system=CONFIRMATION_SYSTEM,
        messages=[
            {
                "role": "user",
                "content": (
                    f"Validated intake record: "
                    f"{record.model_dump_json()}"
                )
            }
        ]
    )

    return response.content[0].text


def ask_for_missing(
    missing_fields: list[str],
    collected_so_far: dict
) -> str:
    """Generate a follow-up question."""

    prompt = (
        f"Fields already collected: "
        f"{json.dumps(collected_so_far)}\n"
        f"Fields still needed: "
        f"{', '.join(missing_fields)}\n"
        "Ask for the first missing field only."
    )

    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        system=FOLLOWUP_SYSTEM,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text


def save_record(
    record: IntakeRecord,
    turns_taken: int
) -> str:
    """Save a log-safe compliance intake record."""

    timestamp = datetime.now().strftime(
        "%Y%m%dT%H%M%S"
    )

    filepath = (
        f"output/intake_{timestamp}.json"
    )

    data = record.model_dump()

    data["timestamp"] = datetime.now().isoformat()
    data["turns_taken"] = turns_taken
    data["log_safe"] = True

    with open(filepath, "w") as f:
        json.dump(
            data,
            f,
            indent=2
        )

    return filepath


def log_debug(message: str):
    """Write to debug.log without raw user text."""

    with open("debug.log", "a") as f:
        f.write(
            f"[{datetime.now().isoformat()}] "
            f"{message}\n"
        )


In [25]:
def run_session(
    initial_message: str,
    additional_turns: list[str] | None = None
):
    """
    Run a SmartIntake AML Compliance session.

    Parameters
    ----------
    initial_message : str
        First user message.

    additional_turns : list[str] | None
        Optional follow-up messages used for
        multi-turn testing.
    """

    all_messages = [initial_message]

    if additional_turns:
        all_messages.extend(additional_turns)

    conversation_history = []
    collected = {}
    turns_taken = 0

    print("\n" + "=" * 60)
    print("SmartIntake — AML Compliance Intake Session")
    print("=" * 60 + "\n")

    for user_input in all_messages:

        turns_taken += 1

        print(f"[Turn {turns_taken}] User: {user_input}")

        conversation_history.append(
            {
                "role": "user",
                "content": user_input
            }
        )

        extracted = extract_with_tool(conversation_history)

        # Non-compliance query / no tool call
        if extracted is None:

            response = client.messages.create(
                model=MODEL,
                max_tokens=200,
                system=EXTRACTION_SYSTEM,
                messages=conversation_history
            )

            reply = response.content[0].text

            print(
                f"[Turn {turns_taken}] "
                f"SmartIntake: {reply}\n"
            )

            conversation_history.append(
                {
                    "role": "assistant",
                    "content": reply
                }
            )

            log_debug(
                f"Turn {turns_taken}: no tool call"
            )

            continue

        # Merge extracted values
        collected.update(
            {
                k: v
                for k, v in extracted.items()
                if v
            }
        )

        try:

            record = IntakeRecord(**collected)

            confirmation = generate_confirmation(record)

            print(
                f"[Turn {turns_taken}] "
                f"SmartIntake: {confirmation}\n"
            )

            filepath = save_record(
                record,
                turns_taken
            )

            log_debug(
                f"Turn {turns_taken}: "
                f"record saved to {filepath}"
            )

            print(
                f"✓ Intake record saved: "
                f"{filepath}"
            )

            print(
                json.dumps(
                    record.model_dump(),
                    indent=2
                )
            )

            return record

        except ValidationError as e:

            missing = [
                err["loc"][0]
                for err in e.errors()
            ]

            log_debug(
                f"Turn {turns_taken}: "
                f"missing fields {missing}"
            )

            followup = ask_for_missing(
                missing,
                collected
            )

            print(
                f"[Turn {turns_taken}] "
                f"SmartIntake: {followup}\n"
            )

            conversation_history.append(
                {
                    "role": "assistant",
                    "content": followup
                }
            )

    print(
        "\nSession ended without "
        "a complete intake record.\n"
    )

    return None

In [26]:
run_session(
    "Transaction Monitoring team here. We have identified a structuring "
    "pattern involving multiple cash deposits across linked accounts in our "
    "US portfolio. A SAR filing with FinCEN is required and the filing "
    "deadline is within the next 24 hours."
)


SmartIntake — AML Compliance Intake Session

[Turn 1] User: Transaction Monitoring team here. We have identified a structuring pattern involving multiple cash deposits across linked accounts in our US portfolio. A SAR filing with FinCEN is required and the filing deadline is within the next 24 hours.
[Turn 1] SmartIntake: # SmartIntake — Compliance Acknowledgement

---

**FORMAL COMPLIANCE INTAKE ACKNOWLEDGEMENT**
**Reference Class:** Regulatory Filing Acknowledgement
**System:** SmartIntake Intake Management Platform

---

**ACKNOWLEDGEMENT OF RECORD**

This acknowledgement confirms receipt and preliminary validation of a formal regulatory filing submission as detailed below:

| Field | Recorded Value |
|---|---|
| **Query Type** | Suspicious Activity Report / Suspicious Transaction Report (SAR/STR) Filing |
| **Regulatory Reference** | Bank Secrecy Act — Anti-Money Laundering (BSA/AML) |
| **Jurisdiction**

✓ Intake record saved: output/intake_20260903T201637.json
{
  "query_type": 

IntakeRecord(query_type='sar_str_filing', regulation_ref='BSA_AML', jurisdiction='usa', urgency='critical', submitting_team='Transaction Monitoring')

In [27]:
run_session(
    initial_message=(
        "Hi, we have a new OFAC sanctions hit in our Singapore business."
    ),
    additional_turns=[
        "The submitting team is Sanctions Operations.",
        "The review must be completed within 24 hours."
    ]
)


SmartIntake — AML Compliance Intake Session

[Turn 1] User: Hi, we have a new OFAC sanctions hit in our Singapore business.
[Turn 1] SmartIntake: What is the filing or response deadline for this matter?

[Turn 2] User: The submitting team is Sanctions Operations.
[Turn 2] SmartIntake: Thank you. What is the filing or response deadline for this sanctions hit, if any? This will help me determine the urgency level.

[Turn 3] User: The review must be completed within 24 hours.
[Turn 3] SmartIntake: # COMPLIANCE ACKNOWLEDGEMENT — FORMAL INTAKE RECORD

---

**Document Reference:** SmartIntake Validated Intake Record
**Acknowledgement Status:** PENDING CONFIRMATION

---

## Intake Record Summary

| Field | Details |
|---|---|
| **Query Type** | Sanctions Match Review |
| **Regulation Reference** | OFAC Specially Designated Nationals (SDN) List |
| **Jurisdiction** | Republic of Singapore |
| **Urgency Tier** | ⚠️ CRITICAL |
| **Submitting Team** | Sanctions Operations |

---

## Formal Ackno

IntakeRecord(query_type='sanctions_match', regulation_ref='OFAC_SDN', jurisdiction='singapore', urgency='critical', submitting_team='Sanctions Operations')

In [28]:
run_session(
    "Transaction Monitoring team here. We have identified a structuring "
    "pattern involving multiple cash deposits across linked accounts in our "
    "US portfolio. A SAR filing with FinCEN is required and the filing "
    "deadline is within the next 24 hours."
)


SmartIntake — AML Compliance Intake Session

[Turn 1] User: Transaction Monitoring team here. We have identified a structuring pattern involving multiple cash deposits across linked accounts in our US portfolio. A SAR filing with FinCEN is required and the filing deadline is within the next 24 hours.
[Turn 1] SmartIntake: # SmartIntake — Compliance Acknowledgement

---

**FORMAL COMPLIANCE INTAKE ACKNOWLEDGEMENT**
**Record Reference:** SAR/STR Filing — BSA/AML
**Generated By:** SmartIntake Automated Intake System

---

**QUERY TYPE:** Suspicious Activity Report / Suspicious Transaction Report (SAR/STR) Filing

**REGULATORY REFERENCE:** Bank Secrecy Act / Anti-Money Laundering (BSA/AML)

**JURISDICTION:** United States of America (USA) — Federal

**URGENCY TIER:** ⚠️ CRITICAL — Immediate escalation and expedited processing required in accordance

✓ Intake record saved: output/intake_20260903T201747.json
{
  "query_type": "sar_str_filing",
  "regulation_ref": "BSA_AML",
  "jurisdiction"

IntakeRecord(query_type='sar_str_filing', regulation_ref='BSA_AML', jurisdiction='usa', urgency='critical', submitting_team='Transaction Monitoring')

In [29]:
run_session(
    "Can you recommend a good science fiction movie for this weekend?"
)



SmartIntake — AML Compliance Intake Session

[Turn 1] User: Can you recommend a good science fiction movie for this weekend?
[Turn 1] SmartIntake: I handle AML and compliance intake only. Please describe your compliance, fraud, sanctions, KYC, or regulatory query.


Session ended without a complete intake record.



In [30]:
run_session(
    initial_message=(
        "Fraud Operations team here. We've detected a pattern of "
        "account takeover attempts on our India payments platform. "
        "It appears to be credential stuffing and needs to be logged "
        "for investigation."
    ),
    additional_turns=[
        "The matter falls under our RBI reporting framework."
    ]
)


SmartIntake — AML Compliance Intake Session

[Turn 1] User: Fraud Operations team here. We've detected a pattern of account takeover attempts on our India payments platform. It appears to be credential stuffing and needs to be logged for investigation.
[Turn 1] SmartIntake: # SmartIntake — Compliance Acknowledgement

---

**FORMAL COMPLIANCE INTAKE ACKNOWLEDGEMENT**
**Reference:** SmartIntake Validated Record
**Document Status:** Pending Confirmation

---

This acknowledgement confirms receipt and preliminary validation of a formally submitted compliance intake record under the parameters detailed below.

**Query Type:** Fraud Investigation
**Applicable Regulation:** Reserve Bank of India — Prevention of Money Laundering Act (RBI_PMLA)
**Jurisdiction:** India
**Urgency Tier:** Standard
**Submitting Team:** Fraud Operations

---

This record has been processed through SmartIntake's automated intake validation pipeline and is staged

✓ Intake record saved: output/intake_20260903T201938.

IntakeRecord(query_type='fraud_investigation', regulation_ref='RBI_PMLA', jurisdiction='india', urgency='standard', submitting_team='Fraud Operations')

In [31]:
from pathlib import Path
import json

output_files = sorted(
    Path("output").glob("intake_*.json")
)

print(f"Records saved: {len(output_files)}\n")

for f in output_files:
    print(f"--- {f.name} ---")

    with open(f) as fp:
        print(
            json.dumps(
                json.load(fp),
                indent=2
            )
        )

    print()

Records saved: 4

--- intake_20260903T201637.json ---
{
  "query_type": "sar_str_filing",
  "regulation_ref": "BSA_AML",
  "jurisdiction": "usa",
  "urgency": "critical",
  "submitting_team": "Transaction Monitoring",
  "timestamp": "2026-09-03T20:16:37.611975",
  "turns_taken": 1,
  "log_safe": true
}

--- intake_20260903T201700.json ---
{
  "query_type": "sanctions_match",
  "regulation_ref": "OFAC_SDN",
  "jurisdiction": "singapore",
  "urgency": "critical",
  "submitting_team": "Sanctions Operations",
  "timestamp": "2026-09-03T20:17:00.089206",
  "turns_taken": 3,
  "log_safe": true
}

--- intake_20260903T201747.json ---
{
  "query_type": "sar_str_filing",
  "regulation_ref": "BSA_AML",
  "jurisdiction": "usa",
  "urgency": "critical",
  "submitting_team": "Transaction Monitoring",
  "timestamp": "2026-09-03T20:17:47.520737",
  "turns_taken": 1,
  "log_safe": true
}

--- intake_20260903T201938.json ---
{
  "query_type": "fraud_investigation",
  "regulation_ref": "RBI_PMLA",
  "jur

In [32]:
print("debug.log contents:")
print("-" * 40)
with open("debug.log") as f:
    print(f.read())



debug.log contents:
----------------------------------------
[2026-09-03T12:25:28.052027] Turn 1: record saved to output/incident_20260903T122528.json
[2026-09-03T12:26:40.735805] Turn 1: no tool call — model replied in text
[2026-09-03T12:26:47.551185] Turn 2: no tool call — model replied in text
[2026-09-03T12:26:58.902625] Turn 3: record saved to output/incident_20260903T122658.json
[2026-09-03T12:27:47.836644] Turn 1: record saved to output/incident_20260903T122747.json
[2026-09-03T12:29:14.367219] Turn 1: no tool call — model replied in text
[2026-09-03T12:36:16.500164] Turn 1: record saved to output/incident_20260903T123616.json
[2026-09-03T12:52:57.262465] Turn 1: record saved to output/incident_20260903T125257.json
[2026-09-03T12:53:10.740526] Turn 1: no tool call — model replied in text
[2026-09-03T12:53:22.038050] Turn 2: no tool call — model replied in text
[2026-09-03T12:53:36.659248] Turn 3: record saved to output/incident_20260903T125336.json
[2026-09-03T12:53:52.213481] 